# Credit-risk baseline

> **Dữ liệu synthetic:** kết quả chỉ phục vụ kiểm tra pipeline, không đại diện cho khách hàng thực.

Notebook này chỉ import logic từ `src/`. Trên Kaggle, clone/upload repository và feature artifacts tạo từ Giai đoạn 4, sau đó chỉnh `PROJECT_ROOT`. Không dùng test trong lúc chọn baseline.

In [1]:
from pathlib import Path
import sys
import yaml
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs" / "config.yaml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "configs" / "config.yaml").is_file():
    raise FileNotFoundError("Không tìm thấy configs/config.yaml từ working directory hiện tại.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.models.train_credit import train_credit_baselines
from src.utils.io import read_parquet_safe

In [2]:
with (PROJECT_ROOT / 'configs/model_credit.yaml').open(encoding='utf-8') as handle:
    credit_config = yaml.safe_load(handle)

feature_dir = PROJECT_ROOT / 'data/features'
X_train = read_parquet_safe(feature_dir / 'X_train.parquet')
X_valid = read_parquet_safe(feature_dir / 'X_valid.parquet')
y_train = read_parquet_safe(feature_dir / 'y_credit_train.parquet')['credit_risk']
y_valid = read_parquet_safe(feature_dir / 'y_credit_valid.parquet')['credit_risk']

{"timestamp": "2026-11-04T15:45:19.095839+00:00", "level": "INFO", "logger": "src.utils.io", "message": "Reading Parquet: e:\\Personal Project\\Data science_analyze finance\\data\\features\\X_train.parquet"}
{"timestamp": "2026-11-04T15:45:19.259143+00:00", "level": "INFO", "logger": "src.utils.io", "message": "Reading Parquet: e:\\Personal Project\\Data science_analyze finance\\data\\features\\X_valid.parquet"}
{"timestamp": "2026-11-04T15:45:19.283944+00:00", "level": "INFO", "logger": "src.utils.io", "message": "Reading Parquet: e:\\Personal Project\\Data science_analyze finance\\data\\features\\y_credit_train.parquet"}
{"timestamp": "2026-11-04T15:45:19.290697+00:00", "level": "INFO", "logger": "src.utils.io", "message": "Reading Parquet: e:\\Personal Project\\Data science_analyze finance\\data\\features\\y_credit_valid.parquet"}


In [4]:
# Set log_mlflow=True when you want local Kaggle MLflow records under mlruns/.
credit_results = train_credit_baselines(
    X_train, y_train, X_valid, y_valid, credit_config, seed=42, log_mlflow=False
)
pd.DataFrame({name: result.metrics for name, result in credit_results.items()}).T

{"timestamp": "2026-11-04T02:33:35.388653+00:00", "level": "INFO", "logger": "src.utils.report", "message": "Wrote HTML report: reports\\credit_baseline_dummy.html"}
{"timestamp": "2026-11-04T02:33:35.390506+00:00", "level": "INFO", "logger": "src.models.train_credit", "message": "Credit baseline dummy validation metrics: {'macro_f1': 0.16828445566216318, 'qwk': 0.0, 'class_0_precision': 0.5073155555555555, 'class_0_recall': 1.0, 'class_0_f1': 0.6731378226486527, 'class_0_support': 57073.0, 'class_1_precision': 0.0, 'class_1_recall': 0.0, 'class_1_f1': 0.0, 'class_1_support': 26164.0, 'class_2_precision': 0.0, 'class_2_recall': 0.0, 'class_2_f1': 0.0, 'class_2_support': 20141.0, 'class_3_precision': 0.0, 'class_3_recall': 0.0, 'class_3_f1': 0.0, 'class_3_support': 9122.0, 'confusion_matrix': [[57073, 0, 0, 0], [26164, 0, 0, 0], [20141, 0, 0, 0], [9122, 0, 0, 0]]}"}
{"timestamp": "2026-11-04T02:33:43.508805+00:00", "level": "INFO", "logger": "src.utils.report", "message": "Wrote HTML re

,macro_f1,qwk,class_0_precision,class_0_recall,class_0_f1,class_0_support,class_1_precision,class_1_recall,class_1_f1,class_1_support,class_2_precision,class_2_recall,class_2_f1,class_2_support,class_3_precision,class_3_recall,class_3_f1,class_3_support,confusion_matrix
dummy,0.168284,0.0,0.507316,1.0,0.673138,57073.0,0.0,0.0,0.0,26164.0,0.0,0.0,0.0,20141.0,0.0,0.0,0.0,9122.0,"[[57073, 0, 0, 0], [26164, 0, 0, 0], [20141, 0..."
logistic_regression,0.59104,0.779409,0.874868,0.756452,0.811362,57073.0,0.425055,0.499656,0.459346,26164.0,0.471706,0.466015,0.468843,20141.0,0.540246,0.740189,0.624607,9122.0,"[[43173, 12120, 1710, 70], [5545, 13073, 6709,..."


## Giai đoạn 6 — LightGBM

Các cell dưới đây chỉ import và gọi lại module trong `src/`. Chạy cell train trên Kaggle khi đã sẵn sàng dùng toàn bộ train/validation; không chạy test tại đây. Để tạo bundle versioned cho inference, dùng `run_training(load_config())`; chỉ dùng `final=True` khi đã chốt mô hình.

In [3]:
from src.models.train_credit import train_credit_lightgbm, train_credit_xgboost
from src.models.predict import load_model_bundle, predict_batch
from src.pipelines.training_pipeline import run_training
from src.utils.config import load_config

In [4]:
# Chạy riêng credit LightGBM: Optuna chỉ dùng train, early stopping dùng validation.
credit_lightgbm = train_credit_lightgbm(
    X_train, y_train, X_valid, y_valid, credit_config, seed=42
)
pd.DataFrame([credit_lightgbm.metrics], index=['lightgbm'])

[I 2026-11-04 22:45:25,091] A new study created in memory with name: no-name-bf07e392-5cb0-4690-a74b-5ec81278771c
c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-11-04 22:47:15,335] Trial 0 finished with value: 0.7726037701945228 and parameters: {'num_leaves': 58, 'max_depth': 12, 'min_child_samples': 123, 'learning_rate': 0.09381218778758511, 'subsample': 0.7468055921327309, 'colsample_bytree': 0.7467983561008608, 'reg_alpha': 2.550264850403287e-06, 'reg_lambda': 1.156732719914599}. Best is trial 0 with value: 0.7726037701945228.
c:\Users\pc\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
[I 2026-11-04 22:48:08,161] Trial 1 finished with value: 0.7724

,macro_f1,qwk,class_0_precision,class_0_recall,class_0_f1,class_0_support,class_1_precision,class_1_recall,class_1_f1,class_1_support,class_2_precision,class_2_recall,class_2_f1,class_2_support,class_3_precision,class_3_recall,class_3_f1,class_3_support,confusion_matrix
lightgbm,0.592347,0.779463,0.879117,0.749636,0.80923,57073.0,0.423886,0.51032,0.463105,26164.0,0.471996,0.466958,0.469464,20141.0,0.544487,0.740627,0.627589,9122.0,"[[42784, 12515, 1709, 65], [5292, 13352, 6720,..."


## XGBoost challenger

XGBoost dùng cùng artifact train/validation với LightGBM và Optuna tối ưu QWK. Đặt `xgboost.device: cuda` trong `configs/model_credit.yaml` khi chạy Kaggle T4; giữ `cpu` khi chạy local không có CUDA.

In [ ]:
credit_xgboost = train_credit_xgboost(
    X_train, y_train, X_valid, y_valid, credit_config, seed=42
)
pd.DataFrame([credit_xgboost.metrics], index=['xgboost'])

In [ ]:
# Sau khi chạy run_training(load_config()) để lưu bundle v1.0.0.
# Bundle inference nhận dữ liệu thô, không phải X_train đã transform.
credit_bundle = load_model_bundle('v1.0.0', 'credit_risk', PROJECT_ROOT / 'models')
raw_preview = pd.read_csv(PROJECT_ROOT / 'data/raw/synthetic_finance_20250901.csv', nrows=10)
predict_batch(raw_preview, credit_bundle)

In [ ]:
# Bundle challenger được tạo bởi: python -m scripts.run_credit_xgboost
xgboost_bundle = load_model_bundle(
    'v1.0.0', 'credit_risk', PROJECT_ROOT / 'models', model_name='xgboost'
)
predict_batch(raw_preview, xgboost_bundle)